# ProductStudio AI - real pipeline + evaluation (Colab)

Runs the **real** SAM2 + SDXL-inpaint + ControlNet pipeline on a free Colab GPU (T4, 16GB) and produces the **real** metric numbers for the resume: mean IoU (Phase 1), CLIP score / product SSIM / product LPIPS / latency (Phase 3).

Why Colab: SDXL needs ~10-12GB VRAM; a local RTX 2050 (4GB) cannot run it. This is the documented, honest fallback.

**Runtime -> Change runtime type -> T4 GPU** before you start.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 1. Install dependencies

In [ ]:
%pip -q install diffusers==0.30.3 transformers==4.44.2 accelerate==0.34.2 \
  controlnet-aux==0.0.9 open-clip-torch==2.26.1 lpips==0.1.4 \
  scikit-image==0.24.0 opencv-python-headless==4.10.0.84 scipy==1.14.1 \
  pydantic==2.9.2 pydantic-settings==2.5.2
%pip -q install "git+https://github.com/facebookresearch/sam2.git"
print('deps installed - if Colab shows a "RESTART SESSION" button, click it, then continue from the GPU-check cell (do NOT re-run this install cell)')

## 2. Get the project code

Option A - clone (if you pushed the repo somewhere): edit and run the git line.

Option B - upload: on your machine zip the `ai-service/` folder (`ai-service.zip`) and upload it here.

In [ ]:
import os
# --- Option A ---
# !git clone https://github.com/<you>/ProductStudio_AI.git && ln -s ProductStudio_AI/ai-service ai-service

# --- Option B ---
if not os.path.isdir('ai-service'):
    from google.colab import files
    up = files.upload()               # choose ai-service.zip
    name = next(iter(up))
    !unzip -qo "{name}"
assert os.path.isdir('ai-service/app/pipelines'), 'expected ai-service/app/pipelines/'
os.chdir('ai-service'); print('cwd =', os.getcwd())

## 3. Download the SAM2 checkpoint

Using `large` here since Colab has the VRAM; drop to `small` if you hit OOM.

In [ ]:
SIZE = 'large'   # tiny | small | base_plus | large
CKPT = {
  'tiny': ('sam2.1_hiera_tiny.pt', 'configs/sam2.1/sam2.1_hiera_t.yaml'),
  'small': ('sam2.1_hiera_small.pt', 'configs/sam2.1/sam2.1_hiera_s.yaml'),
  'base_plus': ('sam2.1_hiera_base_plus.pt', 'configs/sam2.1/sam2.1_hiera_b+.yaml'),
  'large': ('sam2.1_hiera_large.pt', 'configs/sam2.1/sam2.1_hiera_l.yaml'),
}[SIZE]
import os
os.makedirs('models', exist_ok=True)
url = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/' + CKPT[0]
if not os.path.exists('models/' + CKPT[0]):
    !wget -q {url} -O models/{CKPT[0]}
SAM2_CHECKPOINT = 'models/' + CKPT[0]
SAM2_CONFIG = CKPT[1]
print('checkpoint:', SAM2_CHECKPOINT, os.path.getsize(SAM2_CHECKPOINT) // 1_000_000, 'MB')

## 4. Upload your test product images

Use **15-20 real e-commerce product photos** (product roughly centred, plain-ish background works best). These are the real inputs for Phases 1 and 3.

In [ ]:
import os, glob
os.makedirs('eval/test_images', exist_ok=True)
from google.colab import files
for name, data in files.upload().items():
    open('eval/test_images/' + name, 'wb').write(data)
print(sorted(os.path.basename(p) for p in glob.glob('eval/test_images/*')))

## 5. Phase 1 - segmentation IoU

5a generates rough GrabCut masks. **Download `eval/ground_truth/`, hand-correct every mask** in any editor (product = white, background = black), re-upload, then run 5c.

In [ ]:
!python -m eval.make_ground_truth --images eval/test_images --out eval/ground_truth
from google.colab import files; import shutil
shutil.make_archive('ground_truth_rough', 'zip', 'eval/ground_truth'); files.download('ground_truth_rough.zip')

In [ ]:
# 5b - after hand-correcting, re-upload the corrected masks
import os; os.makedirs('eval/ground_truth', exist_ok=True)
from google.colab import files
for name, data in files.upload().items():
    open('eval/ground_truth/' + name, 'wb').write(data)
print('ground-truth masks:', len(os.listdir('eval/ground_truth')))

In [ ]:
!python -m eval.segmentation_iou \
  --images eval/test_images --gt eval/ground_truth \
  --checkpoint {SAM2_CHECKPOINT} --config {SAM2_CONFIG} \
  --device cuda --out eval/results/segmentation_iou.csv

## 6. Phase 2 + 3 - generation + metrics

Edit `eval/prompts.json` first (copy `eval/prompts.example.json`, one entry per uploaded image with a real prompt). Then run:

In [ ]:
import os
if not os.path.exists('eval/prompts.json'):
    import shutil; shutil.copy('eval/prompts.example.json', 'eval/prompts.json')
    print('created eval/prompts.json from the example - EDIT IT to match your images, then re-run this cell')
else:
    !python -m eval.run_eval \
      --images eval/test_images --manifest eval/prompts.json \
      --sam2-checkpoint {SAM2_CHECKPOINT} --sam2-config {SAM2_CONFIG} \
      --device cuda --controlnet canny --out eval/results

## 7. Show results + before/after

In [ ]:
print(open('eval/results/RESULTS.md').read())
import glob, matplotlib.pyplot as plt
from PIL import Image
outs = sorted(glob.glob('eval/results/generations/*__out.png'))[:6]
fig, ax = plt.subplots(len(outs), 3, figsize=(12, 4*len(outs)))
for i, o in enumerate(outs):
    stem = os.path.basename(o).replace('__out.png','')
    src = next(iter(glob.glob(f'eval/test_images/{stem}.*')))
    row = ax[i] if len(outs) > 1 else ax
    for a, (img, t) in zip(row, [(src,'original'), (f'eval/results/generations/{stem}__mask.png','SAM2 mask'), (o,'generated')]):
        a.imshow(Image.open(img)); a.set_title(f'{stem} - {t}'); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import shutil; from google.colab import files
shutil.make_archive('productstudio_eval_results', 'zip', 'eval/results')
files.download('productstudio_eval_results.zip')  # commit eval/results/ back into the repo